In [22]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, log_loss
from sklearn.svm import SVC
from xgboost import XGBClassifier
os.chdir("/home/pgcp-ai/MachineLearning/Cases/Glass_Identification/")

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [3]:
glass = pd.read_csv("Glass.csv")
glass

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,building_windows_float_processed
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,building_windows_float_processed
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,building_windows_float_processed
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,building_windows_float_processed
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,building_windows_float_processed
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,headlamps
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,headlamps
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,headlamps
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,headlamps


In [5]:
le = LabelEncoder()
glass["Type"] = le.fit_transform(glass["Type"])

In [6]:
X, y = glass.drop("Type", axis = 1), glass["Type"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26, stratify = y)

In [17]:
lr = LogisticRegression(random_state=26)
nb = GaussianNB()
dtc = DecisionTreeClassifier(random_state = 26)
rf = RandomForestClassifier(random_state = 26)
stack = StackingClassifier(estimators = [("NB",nb),("LR", lr),("DTC", dtc)],
                       final_estimator = rf)
stack.fit(X_train, y_train)

y_pred = stack.predict(X_test)
y_pred_prob = stack.predict_proba(X_test)
f1_score(y_test, y_pred, average="macro")

/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regressi

0.6754913076341649

In [18]:
log_loss(y_pred,y_pred_prob)

0.4138012884439305

In [19]:
lr = LogisticRegression(random_state=26)
nb = GaussianNB()
dtc = DecisionTreeClassifier(random_state = 26)
rf = RandomForestClassifier(random_state = 26)
stack = StackingClassifier(estimators = [("NB",nb),("LR", lr),("DTC", dtc)],
                       final_estimator = rf,passthrough=True)
stack.fit(X_train, y_train)

y_pred = stack.predict(X_test)
y_pred_prob = stack.predict_proba(X_test)
f1_score(y_test, y_pred, average="macro")

/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regressi

0.6822089947089948

In [26]:
xgbm = XGBClassifier(random_state=26)

In [20]:
log_loss(y_pred,y_pred_prob)

0.40309517511928417

In [29]:
svm = SVC()
stack = StackingClassifier(estimators = [('NB',nb),('RF',rf),('DTC',dtc),('SVM',svm),('lr',lr)],final_estimator=xgbm,passthrough=True)
stack.fit(X_train,y_train)
y_pred = stack.predict(X_test)
y_pred_prob = stack.predict_proba(X_test)
f1_score(y_test,y_pred,average="macro"),log_loss(y_test,y_pred_prob)

/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regressi

(0.6477050169661361, 0.8373897628201392)

In [ ]:
svm = SVC()
stack = StackingClassifier(estimators = [('NB',nb),('RF',rf),('DTC',dtc),('SVM',svm),('lr',lr)],final_estimator=xgbm,passthrough=True)
stack.fit(X_train,y_train)
y_pred = stack.predict(X_test)
y_pred_prob = stack.predict_proba(X_test)
f1_score(y_test,y_pred,average="macro"),log_loss(y_test,y_pred_prob)